In [9]:
from tqdm import tqdm
import numpy as np
from typing import Dict
from model_ranking import (
    ClassificationPredicitonLoadConfig,
    get_classification_pred_path,
    load_h5,
)

In [10]:
path = "/g/kreshuk/talks/sampled_features/classification/mitochondria/EPFL_to_EPFL/DenseNet121_EPFL/Gauss/a001-005/predictions.h5"
consis_score = load_h5(path, "EI_consis")
print(consis_score.shape)

(1, 4680)


In [13]:
def get_classification_consistency_results(config: ClassificationPredicitonLoadConfig, consis_metric_key: str):
        per_target_consis: Dict[str, Dict[str, Dict[str, Dict[str, float]]]] = {}
        for tgt in config.target:
            per_perturbation_consis: Dict[str, Dict[str, Dict[str, float]]] = {}
            for pert_type, pert_levels in config.perturbations.items():
                per_strength_consis: Dict[str, Dict[str, float]] = {}
                for pert_level in tqdm(pert_levels):
                    per_model_consis: Dict[str, float] = {}
                    for src_model in config.source:
                        p_pred_path = get_classification_pred_path(
                            model_name=src_model,
                            target=tgt,
                            base_path=config.base_path,
                            aug=pert_type,
                            aug_str=pert_level,
                        )
                        consis_scores = load_h5(p_pred_path, consis_metric_key)
                        median_score = np.median(consis_scores, axis=1)[0]
                        per_model_consis[src_model] = median_score
                    per_strength_consis[pert_level] = per_model_consis
                per_perturbation_consis[pert_type] = per_strength_consis
            per_target_consis[tgt] = per_perturbation_consis
        return per_target_consis


In [12]:
cfg = ClassificationPredicitonLoadConfig(
    source=[
        "DenseNet121_EPFL",
        "MobileNetV2_EPFL",
        "MobileNetV3_EPFL",
        "ResNet50_EPFL",
        "ResNet18_EPFL",
        "VGG16_EPFL",
    ],
    target=[
        "EPFL",
    ],
    perturbations={
        "Gauss": ["a001-005", "a005-01", "a01-015", "a015-02", "a02-025"],
    },
    base_path="/g/kreshuk/talks/sampled_features/classification/mitochondria",
)
    

In [14]:
consis_scores = get_classification_consistency_results(cfg, "EI_consis")

100%|██████████| 5/5 [00:00<00:00, 12.82it/s]


In [15]:
consis_scores

{'EPFL': {'Gauss': {'a001-005': {'DenseNet121_EPFL': 0.99772227,
    'MobileNetV2_EPFL': 0.96389616,
    'MobileNetV3_EPFL': 0.60946,
    'ResNet50_EPFL': 0.99791425,
    'ResNet18_EPFL': 0.9996587,
    'VGG16_EPFL': 0.9999406},
   'a005-01': {'DenseNet121_EPFL': 0.9968866,
    'MobileNetV2_EPFL': 0.9379638,
    'MobileNetV3_EPFL': 0.6002158,
    'ResNet50_EPFL': 0.9963609,
    'ResNet18_EPFL': 0.9994422,
    'VGG16_EPFL': 0.99991786},
   'a01-015': {'DenseNet121_EPFL': 0.99314225,
    'MobileNetV2_EPFL': 0.8611696,
    'MobileNetV3_EPFL': 0.58775544,
    'ResNet50_EPFL': 0.97916925,
    'ResNet18_EPFL': 0.9983993,
    'VGG16_EPFL': 0.9998549},
   'a015-02': {'DenseNet121_EPFL': 0.96370775,
    'MobileNetV2_EPFL': 0.81013155,
    'MobileNetV3_EPFL': 0.5853758,
    'ResNet50_EPFL': 0.91080177,
    'ResNet18_EPFL': 0.99338114,
    'VGG16_EPFL': 0.9996939},
   'a02-025': {'DenseNet121_EPFL': 0.85952926,
    'MobileNetV2_EPFL': 0.8026845,
    'MobileNetV3_EPFL': 0.5973505,
    'ResNet50_EP